# 01 — Analyse non supervisée des IRM cérébrales

**Projet 7 OpenClassrooms — BrainScanAI (CurelyticsIA)** · Livrable 1 · Étapes 1 à 3 de la mission

## Introduction

### Contexte

CurelyticsIA veut explorer l'automatisation de la détection de tumeurs cérébrales sur des IRM. Sur les images transmises par Clara (responsable Data Science), seules 100 ont été annotées par des radiologues (« cancer » ou « normal ») ; toutes les autres sont sans étiquette.

### Contenu du notebook

Ce notebook constitue le livrable 1 : l'extraction des features, le preprocessing adapté au modèle utilisé, l'analyse non supervisée et l'entraînement de modèles de clustering. L'approche semi-supervisée (étape 4) fait l'objet du notebook 02.

### Séparation des jeux de données

Le jeu « fortement » labellisé (les images annotées par les radiologues) et le jeu « faiblement » labellisé (les labels qui seront issus du clustering) restent séparés dans tout le projet. Ils ne sont jamais mélangés.

### Plan

1. Importer les données et explorer le jeu de radiographies (étape 1) : inventaire des fichiers, tableau des images et des labels, format des images (résolution, canaux de couleur), images d'exemple, qualité et cohérence.
2. Prétraiter les images et extraire les features avec un modèle pré-entraîné (étape 2) : preprocessing adapté au modèle, extraction des embeddings avec les couches convolutionnelles gelées, sauvegarde du tableau de features.
3. Réaliser l'analyse non supervisée (étape 3) : standardisation, réduction de dimension (PCA, t-SNE), clustering à 2 groupes (K-Means, DBSCAN…), contrôle par le score ARI sur les données fortement labellisées, labellisation faible des images sans label.
4. Conclusion.

### Definition of done

✍️ *À compléter.*

## 0. Environnement

Imports nécessaires à l'étape 1. Les bibliothèques de deep learning (PyTorch, torchvision) sont ajoutées au début de l'étape 2, au moment où elles servent. Noyau d'exécution : « Python (Projet 7) » (voir le README du dépôt).

In [1]:
# -------------------------- Imports (étape 1) --------------------------
import sys

sys.path.append("..")  # rend le dossier src/ importable depuis notebooks/

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image  # Pillow : ouvrir les images et lire leurs propriétés (taille, mode de couleur)

from src.settings import FIGURES_PATH, LABELED_PATH, RAW_DATA_PATH, UNLABELED_PATH, random_state

np.random.seed(random_state)  # fixe le hasard : les échantillons tirés seront identiques à chaque exécution
pd.set_option("display.max_colwidth", 60)  # évite de tronquer les noms de dossiers dans les tableaux

print("Dossier des données :", RAW_DATA_PATH)
print("Dossier trouvé :", RAW_DATA_PATH.exists())  # False = le zip n'est pas décompressé dans data/raw/

Dossier des données : /Users/thibaultdautheville/Documents/VSCode/Projet 7/BrainScanAI-semi-supervise/data/raw/mri_dataset_brain_cancer_oc
Dossier trouvé : True


## 1. Importer les données et explorer le jeu de radiographies (étape 1)

Cette étape charge le jeu de données fourni par Clara, explore visuellement les images et vérifie leur résolution, leur canal de couleur et la structure des fichiers. Résultat attendu : une vue d'ensemble du jeu de données, des images d'exemple et des observations sur la qualité et la cohérence des données.

### 1.1 Inventaire des fichiers

Premier contrôle : lister tout ce que contient le zip, par dossier et par type de fichier, avant d'ouvrir la moindre image.

In [ ]:
# -------------------------- 1.1 Inventaire des fichiers --------------------------
# On parcourt tout le dossier du zip sans rien supposer de son contenu : on contrôle d'abord
# « ce qu'on a reçu », pour le comparer ensuite à « ce qui était annoncé » (mail de Clara, documentation).
records = []
for path in sorted(RAW_DATA_PATH.rglob("*")):  # rglob("*") : parcours récursif de tous les sous-dossiers
    if path.is_file() and not path.name.startswith("."):  # ignore les fichiers cachés du système (.DS_Store)
        records.append(
            {
                "dossier": path.parent.relative_to(RAW_DATA_PATH).as_posix(),  # ex. "avec_labels/cancer"
                "extension": path.suffix.lower(),  # .jpg, .png, .txt… (en minuscules pour pouvoir comparer)
                "taille_ko": path.stat().st_size / 1024,  # poids du fichier sur le disque, en kilo-octets
            }
        )
inventory = pd.DataFrame(records)
inventory["dossier"] = inventory["dossier"].replace(".", "(racine du zip)")  # "." = fichier à la racine

# Synthèse : nombre de fichiers et poids moyen, par dossier et par type de fichier
summary = (
    inventory.groupby(["dossier", "extension"])
    .agg(nb_fichiers=("taille_ko", "size"), taille_moyenne_ko=("taille_ko", "mean"))
    .round(1)
)
print(f"{len(inventory)} fichiers au total")
summary

✍️ *À compléter.*